In [1]:
import requests

url = "https://api.fda.gov/device/pma.json"
params = {"search": "advisory_committee:NE", "limit": 5}
r = requests.get(url, params=params).json()

print("Total neuro PMA records:", r["meta"]["results"]["total"])
r["results"][0]

Total neuro PMA records: 3049


{'pma_number': 'P960009',
 'supplement_number': 'S335',
 'applicant': 'Medtronic, Inc.',
 'street_1': '7000 Central Ave., NE',
 'street_2': '',
 'city': 'Minneapolis',
 'state': 'MN',
 'zip': '55432',
 'zip_ext': '',
 'generic_name': 'Stimulator, electrical, implanted, for parkinsonian tremor',
 'trade_name': 'Activa Deep Brian Stimulation Therapy System',
 'product_code': 'MHY',
 'advisory_committee': 'NE',
 'advisory_committee_description': 'Neurology',
 'supplement_type': '30-Day Notice',
 'supplement_reason': 'Process Change - Manufacturer/Sterilizer/Packager/Supplier',
 'expedited_review_flag': 'N',
 'date_received': '2018-12-11',
 'decision_date': '2019-01-08',
 'docket_number': '',
 'decision_code': 'OK30',
 'ao_statement': 'Manufacturing transfer of the M926634A001 Crimp Sleeve, from the current supplier Lake Region Medical to RMS.',
 'openfda': {'registration_number': ['9616693',
   '3006705815',
   '1928237',
   '2649622',
   '3010052483',
   '2648729',
   '3003477135',
   '3

In [2]:
import pandas as pd
import time

records = []
for skip in range(0, 4000, 1000):
    resp = requests.get(url, params={"search": "advisory_committee:NE", "limit": 1000, "skip": skip})
    data = resp.json()
    if "results" not in data:
        break
    records.extend(data["results"])
    time.sleep(0.5)

pma = pd.DataFrame(records)
print(len(pma), "PMA records downloaded")

originals = pma[pma["supplement_number"].fillna("") == ""]
print(len(originals), "original approvals")
originals[["pma_number", "applicant", "trade_name", "decision_date"]].sort_values("decision_date")

3049 PMA records downloaded
43 original approvals


,pma_number,applicant,trade_name,decision_date
1997,P800040,Cordis Corp.,CORDIS PROGRAMMABLE NEURAL STIMULATOR MODELS 9...,1981-04-14
333,P810033,"Avery Biomedical Devices, Inc.",SPINAL EPIDURAL ELECTRODE FOR P.E.N.S. MODEL E...,1982-02-02
58,P820012,Integra Lifesciences Corp.,COSMAN ICP TELE-SENSOR(R),1982-09-02
262,P840001,Medtronic Neuromodulation,ITREL(R) TOTALLY IMPLANTABLE SPINAL CORD STIM....,1984-11-30
2701,P900038,"Sigmedics, Inc.",PARASTEP-I,1994-04-20
610,P970003,"LivaNova USA, Inc.",VNS THERAPY SYSTEM,1997-07-16
1057,P960009,"Medtronic, Inc.",MEDTRONIC ACTIVA TREMOR CONTROL SYSTEM,1997-07-31
1867,P960057,Wrightmedicaltechnologyinc,ADCON-L ANTI-ADHESION CONTROL IN A BARRIER GEL,1998-05-27
3015,P960034,"Abbott Medical Optics, Inc.",CEEON HEPARIN SURFACE MODIFIED (ASM) ULTRAVIOL...,1998-08-12
451,P960033,Staar Sugical,STAARVISC TM SODIUM HYALURONATE,1999-07-02


In [3]:
k_url = "https://api.fda.gov/device/510k.json"

r = requests.get(k_url, params={"search": "advisory_committee:NE", "limit": 1}).json()
print("Neuro 510(k) records:", r["meta"]["results"]["total"])

codes = requests.get(k_url, params={"search": "advisory_committee:NE", "count": "decision_code"}).json()
pd.DataFrame(codes["results"])

Neuro 510(k) records: 6133


KeyError: 'results'

In [4]:
codes

{'error': {'code': 'SERVER_ERROR',
  'message': 'Check your request and try again',
  'details': '[illegal_argument_exception] Text fields are not optimised for operations that require per-document field data like aggregations and sorting, so these operations are disabled by default. Please use a keyword field instead. Alternatively, set fielddata=true on [decision_code] in order to load field data by uninverting the inverted index. Note that this can use significant memory.'}}

In [5]:
k_records = []
for skip in range(0, 7000, 1000):
    resp = requests.get(k_url, params={"search": "advisory_committee:NE", "limit": 1000, "skip": skip})
    data = resp.json()
    if "results" not in data:
        break
    k_records.extend(data["results"])
    time.sleep(0.5)

k510 = pd.DataFrame(k_records)
print(len(k510), "510(k) records downloaded")
k510["decision_code"].value_counts()

6133 510(k) records downloaded


decision_code
SESE    6062
DENG      52
SN        12
SESU       3
SESP       2
SESD       1
SESK       1
Name: count, dtype: int64